In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
from risk_validation.core.metrics.impl import pd as pd_metrics  # noqa: F401
from risk_validation.core.services.pd.metrics_service import PDMetricsService
from risk_validation.core.utils.plots import plot_roc, plot_gains, plot_ks_cdf_with_maxgap

# Load data
DATA_DIR = Path('..') / 'data'
df = pd.read_csv(DATA_DIR / 'sample.csv')

y_col = 'default_flag'
p_col = 'pred_br'
score_col = 'score_pd'
period_col = 'QTR'

# Filter data based on dev and val years
df_dev = df[df['score_year'] == 2019].copy()
df_val = df[df['score_year'] == 2020].copy()

# Initialize the PDMetricsService with the required metrics
service = PDMetricsService(['gini', 'auc_roc', 'ks'])

# Set up parameters for computation
params = {
    'y_col': y_col,
    'p_col': p_col,
    'score_col': score_col,
}

# Compute metrics for Development data
dev_result = service.compute(df_dev, params)
print("Development Metrics:", dev_result)

# Compute metrics for Validation data
val_result = service.compute(df_val, params)
print("Validation Metrics:", val_result)

# Plotting ROC, Gini, and KS
plot_dir = Path('.') / 'model_reports' / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

# ROC Curve
gini_plot_path = plot_gains(
    y_true=df_val[y_col],
    y_score=df_val[p_col],
    title='Gini / Lorenz Curve - Validation Data',
    out_path=plot_dir / 'gini_curve_val.png'
)
display(Image(filename=str(gini_plot_path)))

# AUC-ROC Curve
roc_plot_path = plot_roc(
    y_true=df_val[y_col],
    y_score=df_val[p_col],
    title='ROC Curve - Validation Data',
    out_path=plot_dir / 'roc_curve_val.png'
)
display(Image(filename=str(roc_plot_path)))

# KS CDF Plot
ks_plot_path, _ = plot_ks_cdf_with_maxgap(
    y_true=df_val[y_col],
    y_score=df_val[p_col],
    title='KS CDF Plot',
    out_path=plot_dir / 'ks_cdf_val.png'
)
display(Image(filename=str(ks_plot_path)))